# 제품 묶음 번호(LOT_NO)가 언제 바뀌는지 확인하기

**담당: 김정렬**  ·  관련 Issue: #10 (번호를 채우세요)

## 이 노트북에서 할 일

온도 CSV의 `LOT_NO`(제품 묶음 번호)가 **몇 종류나 있는지**, **언제 바뀌는지**,
한 묶음이 **보통 몇 분 동안 이어지는지** 확인합니다.

## 왜 하는지

공장은 같은 규격 제품을 한동안 이어서 만듭니다. 그 한 묶음이 LOT입니다.
LOT_NO가 바뀌었다는 건 다른 제품으로 넘어갔다는 뜻이고, 조건이 바뀌니 온도도 당연히 달라집니다.

이걸 모르고 분석하면 **정상적인 제품 교체를 "설비 이상"으로 잡아버립니다.**
그래서 LOT이 바뀌는 지점에서 데이터를 끊어서 봐야 합니다.

## 진행 방법

1. 아래 칸을 **위에서부터 순서대로** 실행하세요.
2. `# TODO` 라고 적힌 곳을 직접 채우세요. 막히면 디스코드에 물어보세요.
3. 맨 아래 **결과 정리** 칸에 확인한 값을 한국어 문장으로 적으세요. 이게 진짜 결과물입니다.
4. 다 했으면 커밋 전에 맨 마지막 안내를 읽으세요.

> 데이터 파일이 없으면 `data/` 폴더에 CSV 3개를 먼저 넣으세요. 이 파일들은 Git에 올라가지 않습니다(공유받은 자료를 각자 직접 넣습니다).

## 공통 준비

In [2]:
# 이 칸은 그대로 실행하세요. 데이터 경로를 잡아둡니다.
import os
import pandas as pd

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
TEMP_CSV   = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV  = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")

pd.set_option("display.max_columns", 50)
print("경로 확인:", os.path.exists(TEMP_CSV), os.path.exists(EVENT_CSV), os.path.exists(REPAIR_CSV))

경로 확인: True True True


### 1단계 — 데이터를 읽고 정렬하세요

In [ ]:
df = pd.read_csv(TEMP_CSV, encoding="utf-8")

# TODO: MEAS_DT를 날짜/시간 형식으로 바꾸고 시간순 정렬하세요.

print(df.shape)
print(df["LOT_NO"].head())

df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"])
df = df.sort_values("MEAS_DT").reset_index(drop=True)



(345390, 24)
0    240001
1    240001
2    240001
3    240001
4    240001
Name: LOT_NO, dtype: int64


### 2단계 — LOT_NO가 몇 종류인지 세어 보세요

In [12]:
# TODO: LOT_NO 의 종류 개수를 출력하세요.
# 힌트: df["LOT_NO"].nunique()
print(df["LOT_NO"].nunique())

# TODO: 각 LOT_NO 가 몇 행씩 있는지도 확인해 보세요.
# 힌트: df["LOT_NO"].value_counts().head(10)
print(df["LOT_NO"].value_counts())

14
LOT_NO
240069    46836
240001    40402
240121    38237
240091    37633
240037    35348
240024    31729
240108    26982
240117    25708
240114    18763
240061    13299
240058    12559
240038     7342
240133     6218
240044     4334
Name: count, dtype: int64


### 3단계 — LOT별로 시작 시각, 끝 시각, 지속 시간을 구하세요

`groupby` 를 쓰면 LOT_NO별로 묶어서 계산할 수 있습니다.

In [14]:
# TODO: LOT_NO 별로 MEAS_DT의 최소/최대/개수를 구하세요.
# 힌트:
# lot = df.groupby("LOT_NO")["MEAS_DT"].agg(["min", "max", "count"])
# lot["duration_min"] = (lot["max"] - lot["min"]).dt.total_seconds() / 60
# lot.head(10)
lot = df.groupby("LOT_NO")["MEAS_DT"].agg(["min", "max", "count"])
lot["duration_min"] = (lot["max"] - lot["min"]).dt.total_seconds() / 60
print(lot)

                       min                 max  count  duration_min
LOT_NO                                                             
240001 2024-01-02 03:36:00 2024-01-02 15:04:30  40402    688.500000
240024 2024-03-07 21:22:00 2024-03-08 06:29:48  31729    547.800000
240037 2024-04-13 23:16:00 2024-04-14 09:27:19  35348    611.316667
240038 2024-04-16 11:35:00 2024-04-16 13:39:56   7342    124.933333
240044 2024-05-03 18:50:00 2024-05-03 20:04:38   4334     74.633333
240058 2024-06-09 05:26:00 2024-06-09 09:02:53  12559    216.883333
240061 2024-06-15 01:04:00 2024-06-15 04:49:23  13299    225.383333
240069 2024-07-07 16:57:00 2024-07-08 06:29:34  46836    812.566667
240091 2024-09-08 22:03:00 2024-09-09 08:44:12  37633    641.200000
240108 2024-10-18 04:05:00 2024-10-18 11:50:53  26982    465.883333
240114 2024-11-04 15:40:00 2024-11-04 20:59:02  18763    319.033333
240117 2024-11-13 07:08:00 2024-11-13 14:32:19  25708    444.316667
240121 2024-11-24 07:31:00 2024-11-24 18:22:12  

### 4단계 — 지속 시간의 분포를 보세요

`describe()` 를 쓰면 평균·최솟값·최댓값을 한 번에 볼 수 있습니다.

In [15]:
# TODO: duration_min 의 분포를 확인하세요.
# 힌트: lot["duration_min"].describe()
lot["duration_min"].describe()

count     14.000000
mean     423.646429
std      244.542993
min       74.633333
25%      219.008333
50%      455.100000
75%      633.729167
max      812.566667
Name: duration_min, dtype: float64

### 5단계 — 같은 LOT_NO가 중간에 다시 나타나는지 확인하세요

만약 LOT_NO가 `A, A, A, B, B, A, A` 처럼 **떨어져서 다시 나오면**,
"LOT_NO가 같다 = 연속된 한 구간" 이라는 가정이 깨집니다. 이건 꼭 확인해야 합니다.

In [25]:
# TODO: LOT_NO 가 바뀌는 지점마다 번호를 새로 매겨서, 구간 개수와 LOT 종류 수를 비교해 보세요.
# 힌트:
# block = (df["LOT_NO"] != df["LOT_NO"].shift()).cumsum()
# print("구간 개수:", block.nunique(), " / LOT 종류 수:", df["LOT_NO"].nunique())
# 두 숫자가 같으면 -> 각 LOT은 한 번씩만 연속으로 나옴 (가정 성립)
# 구간 개수가 더 많으면 -> 같은 LOT_NO가 떨어져서 다시 나온다는 뜻
block = (df["LOT_NO"] != df["LOT_NO"].shift()).cumsum()
print("구간 개수:", block.nunique(), " / LOT 종류 수:", df["LOT_NO"].nunique())

구간 개수: 14  / LOT 종류 수: 14


## 결과 정리 — 여기를 꼭 채우세요

아래 빈칸을 확인한 값으로 바꿔서 적으세요. 이 내용이 `docs/02-data-contract.md`로 옮겨집니다.

| 항목 | 확인한 값 |
| --- | --- |
| LOT_NO 종류 수 | 14 개 |
| 한 LOT의 평균 지속 시간 | 약 423.6 분 |
| 가장 짧은 LOT | 74.6 분 |
| 가장 긴 LOT | 812.6 분 |
| LOT 하나당 평균 행 수 | 24670.7 행 |
| 같은 LOT_NO가 떨어진 위치에서 다시 나타나는가 | 아니오 |


### 이상하다고 느낀 점 / 확실하지 않은 점

- 최단 LOT와 최장 LOT가 약 10배 이상 차이 나기 때문에, LOT마다 실제 생산시간이 다른 것인지, 아니면 LOT 변경 시점이나 데이터 수집 방식 때문에 차이가 발생한 것인지 확인이 필요합니다.

---

## 커밋하기 전에 읽으세요

1. **출력은 지우지 않아도 됩니다.** `nbstripout` 필터를 등록해 뒀다면 `git add` 할 때 자동으로 지워집니다.
   등록했는지 확인: `python -m nbstripout --status` → `Automatic cleanup enabled` 가 나와야 합니다.
   안 나오면: `python -m nbstripout --install --attributes .gitattributes`
2. 이 노트북 **한 파일만** 커밋하세요. 다른 사람 파일은 건드리지 마세요.
3. 브랜치를 만들어서 작업하세요. 예) `git switch -c feat/이슈번호-설명`
4. 커밋 메시지 첫 줄: `feat: LOT 단위 구간과 지속 시간 확인`
5. PR 본문에 `Closes #이슈번호` 를 꼭 적으세요.